# 05 — Stratified sampling

Selects a balanced subset of intersections for annotation/reprojection.

**Inputs**
- `data/processed/intersections_stratified.gpkg` — intersections with stratum labels (notebook 03)
- `data/processed/leg_photo_selection_{IMAGE_MODE}.csv` — one row per intersection leg with selected photo (notebook 04)

**Output**
- `data/processed/sampled_legs_{IMAGE_MODE}.csv` — one row per sampled intersection; one randomly selected leg (and its photo) per intersection; feeds into notebook 06

Set `IMAGE_MODE = "panorama"` or `"directional"` in the config cell to match the notebook 04 run.
Strata dimensions and their live unique values are printed after the data is loaded (see section 2).

In [28]:
import os

# Base project directory
PROJECT_DIR = r"C:\Users\Thijs\OneDrive\Documents\Studie\EPA\Second_year\Afstuderen\Project\intersections"

# --- Image mode ---
# Must match the IMAGE_MODE used when running notebooks 02 and 04.
# "panorama"   : one 360° image per location
# "directional": four shots per location
IMAGE_MODE = "directional"  # change to "directional" to process four-direction photos

# --- Input files ---
INT_FILE  = os.path.join(PROJECT_DIR, "data", "processed", "intersections_stratified.gpkg")
# Leg file reflects IMAGE_MODE — must match the run that produced leg_photo_selection_{IMAGE_MODE}.csv
LEGS_FILE = os.path.join(PROJECT_DIR, "data", "processed",
                         f"leg_photo_selection_{IMAGE_MODE}.csv")

# --- Output file ---
# Filename reflects IMAGE_MODE so panorama and directional samples don't overwrite each other.
OUTPUT_FILE = os.path.join(PROJECT_DIR, "data", "processed",
                           f"sampled_legs_{IMAGE_MODE}.csv")

# --- Sampling strategy ---
# 'equal'        : sample exactly N_PER_STRATUM intersections from each stratum
#                  (or all available if fewer than N_PER_STRATUM exist)
# 'proportional' : sample proportionally to stratum size, total = N_TOTAL
SAMPLING_STRATEGY = "equal"

# Used when SAMPLING_STRATEGY = 'equal'
N_PER_STRATUM = 70

# Used when SAMPLING_STRATEGY = 'proportional'
N_TOTAL = 500

# Reproducibility seed
RANDOM_SEED = 42

# --- Minimum photo distance filter ---
# When selecting one leg per intersection, legs whose photo is closer than this
# distance (metres) are excluded from the draw — so very close shots (e.g. 9m, 12m)
# caused by sparse coverage near the intersection are not picked when a better leg exists.
# Fallback: if ALL legs for an intersection are below this threshold, the closest
# available leg is kept anyway so the intersection is not lost from the sample.
MIN_PHOTO_DIST_M = 15  # metres — set to 0 to disable the filter

# All possible stratum dimension columns (mirrors the USE_* toggles in notebook 03).
# DIM_COLS is resolved automatically after loading the file — no need to edit this.
ALL_DIM_COLS = ["dim_type", "dim_risk", "dim_priority", "dim_speed"]

print(f"IMAGE_MODE        : {IMAGE_MODE}")
print(f"LEGS_FILE         : {os.path.basename(LEGS_FILE)}")
print(f"OUTPUT            : {os.path.basename(OUTPUT_FILE)}")
print(f"Strategy          : {SAMPLING_STRATEGY}")
if SAMPLING_STRATEGY == "equal":
    print(f"N per stratum     : {N_PER_STRATUM}")
else:
    print(f"N total           : {N_TOTAL}")
print(f"MIN_PHOTO_DIST_M  : {MIN_PHOTO_DIST_M}m")
print("(DIM_COLS will be resolved after loading intersections_stratified.gpkg)")

IMAGE_MODE        : directional
LEGS_FILE         : leg_photo_selection_directional.csv
OUTPUT            : sampled_legs_directional.csv
Strategy          : equal
N per stratum     : 70
MIN_PHOTO_DIST_M  : 15m
(DIM_COLS will be resolved after loading intersections_stratified.gpkg)


## 1. Load data

In [29]:
import pandas as pd
import geopandas as gpd

# Load strata labels — one row per intersection
intersections = gpd.read_file(INT_FILE)
print(f"Intersections loaded : {len(intersections):,}")

# Resolve DIM_COLS automatically: keep only the possible dim columns
# that are actually present in the loaded file. This mirrors whichever
# USE_* toggles were active when notebook 03 was last run.
DIM_COLS = [c for c in ALL_DIM_COLS if c in intersections.columns]
missing  = [c for c in ALL_DIM_COLS if c not in intersections.columns]
print(f"Active dimensions : {DIM_COLS}")
if missing:
    print(f"Skipped (not in file, USE_* was False in notebook 03): {missing}")
if not DIM_COLS:
    raise ValueError(
        "No stratum columns found in the intersections file.\n"
        "Re-run notebook 03 to regenerate intersections_stratified.gpkg."
    )

# Load leg photo selections — one row per approach leg
legs = pd.read_csv(LEGS_FILE)
print(f"\nLegs loaded : {len(legs):,}")
print(f"Unique intersections in legs : {legs['intersection_id'].nunique():,}")

Intersections loaded : 4,715
Active dimensions : ['dim_type', 'dim_risk', 'dim_priority', 'dim_speed']

Legs loaded : 12,677
Unique intersections in legs : 4,615


## 2. Stratum distribution (all intersections)

Full breakdown of all intersections in `intersections_stratified.gpkg`, regardless of photo coverage.
Use this to understand why certain strata are small — e.g. `T / hoog / VRI / 50+` is rare because
high-risk T-junctions with a traffic light are uncommon by definition.

In [30]:
# Full stratum distribution from the stratified file — all 4,858 intersections,
# regardless of whether a photo was found for them.
all_strata = (
    intersections
    .groupby(DIM_COLS)
    .size()
    .reset_index(name="n_total")
    .sort_values(DIM_COLS)
)

total = all_strata["n_total"].sum()
all_strata["pct"] = (all_strata["n_total"] / total * 100).round(1)

print(f"Total intersections : {total:,}")
print(f"Unique strata       : {len(all_strata)}\n")
print(all_strata.to_string(index=False))

Total intersections : 4,715
Unique strata       : 6

dim_type dim_risk  dim_priority dim_speed  n_total  pct
      4+        _           VRI         _      168  3.6
      4+        _ geen_voorrang         _      699 14.8
      4+        _      voorrang         _      384  8.1
       T        _           VRI         _       18  0.4
       T        _ geen_voorrang         _     2894 61.4
       T        _      voorrang         _      552 11.7


## 3. Join strata to legs and check coverage

Not every intersection in the stratified file will have legs — notebook 04 only assigns
photos to intersections that have at least one panorama within range. We sample only
from the covered intersections.

In [31]:
import pandas as pd
import numpy as np

# Build a per-intersection summary: how many legs, and which stratum
covered = (
    legs
    .groupby("intersection_id")
    .size()
    .reset_index(name="n_legs")
)

# Include is_centrum if nb03 added it to intersections_stratified.gpkg.
# This allows nb06 to use centrum status for attribute-balanced pair selection.
extra_cols = [c for c in ["is_centrum"] if c in intersections.columns]

# Join strata labels (and optional extra columns) from the stratified intersections file
covered = covered.merge(
    intersections[["JTE_ID"] + DIM_COLS + extra_cols],
    left_on="intersection_id",
    right_on="JTE_ID",
    how="left"
).drop(columns="JTE_ID")

if extra_cols:
    print(f"Extra columns passed through: {extra_cols}")

n_matched = covered[DIM_COLS].notna().all(axis=1).sum()
n_unmatched = len(covered) - n_matched
print(f"Intersections with photo coverage : {len(covered):,}")
print(f"  of which have stratum labels    : {n_matched:,}")
if n_unmatched > 0:
    print(f"  WARNING: {n_unmatched} intersections have no stratum match.")
    print("  This usually means notebook 04 was run in test mode (N_INTERSECTIONS=50)")
    print("  on a different set of intersections than what notebook 03 produced.")
    print("  Unmatched intersections will be excluded from sampling.")

# Drop intersections with no stratum label — they cannot be assigned to a stratum
covered = covered.dropna(subset=DIM_COLS)

# Show stratum sizes (only intersections with coverage)
stratum_counts = (
    covered
    .groupby(DIM_COLS, dropna=False)
    .size()
    .reset_index(name="n_available")
    .sort_values(DIM_COLS)
)
print(f"\nStratum coverage ({len(stratum_counts)} strata with at least 1 intersection):")
print(stratum_counts.to_string(index=False))

Extra columns passed through: ['is_centrum']
Intersections with photo coverage : 4,615
  of which have stratum labels    : 4,615

Stratum coverage (6 strata with at least 1 intersection):
dim_type dim_risk  dim_priority dim_speed  n_available
      4+        _           VRI         _          167
      4+        _ geen_voorrang         _          682
      4+        _      voorrang         _          384
       T        _           VRI         _           18
       T        _ geen_voorrang         _         2821
       T        _      voorrang         _          543


In [32]:
# Print the live unique values for each stratum dimension,
# so you can see what categories are actually present in the data.
print("Stratum dimension values (from intersections_stratified.gpkg):\n")
for col in DIM_COLS:
    vals = sorted(covered[col].dropna().unique(), key=str)
    print(f"  {col:12s}: {vals}")

Stratum dimension values (from intersections_stratified.gpkg):

  dim_type    : ['4+', 'T']
  dim_risk    : ['_']
  dim_priority: ['VRI', 'geen_voorrang', 'voorrang']
  dim_speed   : ['_']


## 4. Sample intersections

In [33]:
if SAMPLING_STRATEGY == "equal":
    # Sample up to N_PER_STRATUM intersections from each stratum.
    # Iterate explicitly to avoid pandas version-dependent groupby+apply index behaviour.
    sampled_parts = []
    for _, group in covered.groupby(DIM_COLS):
        sampled_parts.append(
            group.sample(min(len(group), N_PER_STRATUM), random_state=RANDOM_SEED)
        )
    sampled = pd.concat(sampled_parts, ignore_index=True)

elif SAMPLING_STRATEGY == "proportional":
    # Sample proportionally to stratum size, rounding to nearest int, total ≈ N_TOTAL.
    sampled_parts = []
    total_covered = len(covered)
    for _, group in covered.groupby(DIM_COLS):
        n = max(1, round(len(group) / total_covered * N_TOTAL))
        sampled_parts.append(
            group.sample(min(len(group), n), random_state=RANDOM_SEED)
        )
    sampled = pd.concat(sampled_parts, ignore_index=True)

else:
    raise ValueError(f"Unknown SAMPLING_STRATEGY: {SAMPLING_STRATEGY!r}. Use 'equal' or 'proportional'.")

print(f"Sampled intersections : {len(sampled):,}")
print(f"\nPer-stratum sample counts:")
print(
    sampled
    .groupby(DIM_COLS)
    .size()
    .reset_index(name="n_sampled")
    .merge(stratum_counts, on=DIM_COLS)
    .to_string(index=False)
)

Sampled intersections : 368

Per-stratum sample counts:
dim_type dim_risk  dim_priority dim_speed  n_sampled  n_available
      4+        _           VRI         _         70          167
      4+        _ geen_voorrang         _         70          682
      4+        _      voorrang         _         70          384
       T        _           VRI         _         18           18
       T        _ geen_voorrang         _         70         2821
       T        _      voorrang         _         70          543


## 5. Pull legs for sampled intersections

In [34]:
# Filter leg file to only the sampled intersections
sampled_legs = legs[legs["intersection_id"].isin(sampled["intersection_id"])].copy()

# Attach stratum labels so the output is self-contained.
# Also pass through is_centrum if it was carried by covered -> sampled.
extra_cols = [c for c in ["is_centrum"] if c in sampled.columns]
sampled_legs = sampled_legs.merge(
    sampled[["intersection_id"] + DIM_COLS + extra_cols],
    on="intersection_id",
    how="left"
)

if extra_cols:
    print(f"Extra columns included in output: {extra_cols}")

print(f"All legs before selection : {len(sampled_legs):,} (avg {len(sampled_legs) / sampled['intersection_id'].nunique():.1f} per intersection)")
print(sampled_legs.head())

Extra columns included in output: ['is_centrum']
All legs before selection : 1,010 (avg 2.7 per intersection)
   intersection_id  leg_bearing photo_filename  photo_dist_m  u_deg  \
0        176268014        197.3      364_03471          18.9 -164.6   
1        176268014        290.5      364_02668          21.9  -76.6   
2        176268028        151.7      364_03513          18.4  149.5   
3        176268028        236.6      364_03521          22.2 -106.9   
4        176268028         22.9      364_02642          13.3   17.5   

     photo_x     photo_y  photo_bearing_to_inter  candidate_rank  is_panorama  \
0  88139.448  434091.576                   195.4               1        False   
1  88155.684  434068.292                   283.4               1        False   
2  88271.734  434050.359                   149.5               1        False   
3  88302.291  434040.994                   253.1               1        False   
4  88277.047  434021.805                    17.5          

## 5b. Select one representative leg per intersection

Each sampled intersection has multiple approach legs (typically 2–4). A single leg is
chosen at random — seeded by `RANDOM_SEED` — to represent the intersection in the BT
survey. This gives each intersection exactly one photo, which is what notebook 06 expects.

In [35]:
# Apply minimum distance filter before random leg selection.
# Legs with photo_dist_m < MIN_PHOTO_DIST_M are excluded from the draw so that
# very close photos (e.g. 9m, 12m) are only picked as a last resort.
# Fallback: intersections where ALL legs are below the threshold keep their legs
# so the intersection is not lost from the sample.
if MIN_PHOTO_DIST_M > 0:
    qualified   = sampled_legs[sampled_legs["photo_dist_m"] >= MIN_PHOTO_DIST_M]
    inters_qual = set(qualified["intersection_id"])
    inters_all  = set(sampled_legs["intersection_id"])
    inters_fallback = inters_all - inters_qual  # intersections with no qualifying leg

    if inters_fallback:
        # Keep all legs for these intersections so at least one row per intersection exists
        fallback_legs = sampled_legs[sampled_legs["intersection_id"].isin(inters_fallback)]
        legs_for_selection = pd.concat([qualified, fallback_legs])
        print(f"Intersections using fallback (all legs < {MIN_PHOTO_DIST_M}m): {len(inters_fallback)}")
    else:
        legs_for_selection = qualified

    n_removed = len(sampled_legs) - len(legs_for_selection)
    print(f"Legs removed by MIN_PHOTO_DIST_M={MIN_PHOTO_DIST_M}m filter: {n_removed} / {len(sampled_legs)}")
    print(f"Legs remaining for selection: {len(legs_for_selection):,}")
else:
    legs_for_selection = sampled_legs
    print("MIN_PHOTO_DIST_M=0 — distance filter disabled, all legs eligible")

# Select one leg at random per intersection using a fixed seed for reproducibility.
# groupby + sample(1) guarantees exactly one row per intersection; RANDOM_SEED controls
# which leg is chosen so the selection is stable across re-runs.
sampled_legs = (
    legs_for_selection
    .groupby("intersection_id")
    .sample(1, random_state=RANDOM_SEED)
    .reset_index(drop=True)
)

print(f"\nAfter random leg selection : {len(sampled_legs):,} rows (one per intersection)")
print(f"\nPhoto distance stats of selected legs:")
print(sampled_legs["photo_dist_m"].describe(percentiles=[.05, .25, .5, .75, .95]).round(1))
print(f"\nLegs below {MIN_PHOTO_DIST_M}m in final selection (fallback): "
      f"{(sampled_legs['photo_dist_m'] < MIN_PHOTO_DIST_M).sum()}")

Intersections using fallback (all legs < 15m): 1
Legs removed by MIN_PHOTO_DIST_M=15m filter: 41 / 1010
Legs remaining for selection: 969



After random leg selection : 368 rows (one per intersection)

Photo distance stats of selected legs:
count    368.0
mean      21.7
std        3.1
min       13.9
5%        17.8
25%       19.5
50%       21.4
75%       22.9
95%       27.0
max       35.6
Name: photo_dist_m, dtype: float64

Legs below 15m in final selection (fallback): 1


## 6. Save

In [36]:
sampled_legs.to_csv(OUTPUT_FILE, index=False)
print(f"Saved {len(sampled_legs):,} rows to {OUTPUT_FILE}")

Saved 368 rows to C:\Users\Thijs\OneDrive\Documents\Studie\EPA\Second_year\Afstuderen\Project\intersections\data\processed\sampled_legs_directional.csv
